# TSARA on the campaign archive — alignment on real data

**Phase 4, the companion to notebook 04.** Notebook 04 runs every alignment
operation on data manufactured inside it, where the truth is known. This notebook
runs the same operations on the real campaign archive, where there is no answer
key, so you can see them work on the records they were built for. Notebook 04
knows the true atmosphere behind every result, because it made it; nothing here
does, and that asymmetry is the reason the generated notebook exists.

Without an answer key, the evidence is of three other kinds:

* **✔ checks against a loop written from the definition**, as in notebook 04.
  They hold on *any* drive day and at *any* parameters, so change them freely;
* **the numbers `docs/METHODS.md` §11 quotes**, re-measured here from the files
  and collected in a **ledger** (section 12). A number is only *compared* when
  your parameters match the ones METHODS measured under; otherwise the ledger
  shows it and says so;
* **physical plausibility**, in the figures.

---

## How to use this notebook

1. **Point it at the archive.** Set `TSARA_ARCHIVE` to the directory holding the
   archive's `2024/` and `2026/` trees, then start Jupyter from that shell:

   ```bash
   export TSARA_ARCHIVE=/path/to/Data
   ```

2. **Run everything once** (*Run All*, about half a minute).
3. **Two loading cells do the slow part.** *Load one drive* reads a single drive
   day (sections 1–6 use it); *Load ten drives* reads all ten 2024 drive days
   (sections 7–10 use it). Their parameters, such as `DRIVE_DAY`, need a re-run
   of that loading cell and then of the sections below it.
4. **Every section has its own parameters cell.** Change a value and re-run from
   that cell to the next heading; nothing else needs re-running.
5. **"Try it"** notes suggest changes and say what should happen on the archive.

It reads only `2024/NOAA_MobileLab_Drives/` (ten drive days), the University of
Wyoming GPS logs, and `2026/03_instrument_aligned/` (the LANL van), and writes
only to a temporary directory. Of the six-institution `SLC-SOS/2024_mobile/`
hierarchy -- the tree the manifest's `{institution}` path template exists for,
and where the archive's awkward delimited-text formats live -- it touches
nothing but those GPS logs. That is a choice of two well-understood platforms,
not a survey of the archive.

**It is committed without outputs**, because its outputs are the archive's data.

---
## Setup

The toolkit: the archive gate, imports, the figure style shared by notebooks
01–04, the reference implementations the ✔ checks compare TSARA against, and the
ledger that compares re-measured numbers with `METHODS.md`.

In [ ]:
# =============================================================================
# TOOLKIT. Every section needs this cell.
# =============================================================================
import logging
import os
import tempfile
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tsara import setup_logging
from tsara.align import (
    TsaraAlignError,
    attach_positions,
    bin_streams_onto_cells,
    build_output_grid,
    grid_cells,
    interpolate_onto_cells,
    load_grid,
    pair_species,
    save_grid,
)
from tsara.config.analysis import OutputGridConfig
from tsara.config.loader import load_manifest
from tsara.core.naming import sigma_rand_name, sigma_sys_name
from tsara.core.support import CellBounds
from tsara.ingest import ingest_campaign

# --- the archive, and only the parts of it this notebook may read -------------
ARCHIVE_ENV = "TSARA_ARCHIVE"
if not os.environ.get(ARCHIVE_ENV):
    raise RuntimeError(
        f"This notebook reads the campaign archive, and {ARCHIVE_ENV} is not set. Set it to "
        "the directory holding the archive's 2024/ and 2026/ trees, e.g. "
        f"`export {ARCHIVE_ENV}=/path/to/Data`, start Jupyter from that shell, and run again. "
        "Notebook 04 runs the same operations on generated data and needs nothing."
    )
ARCHIVE = Path(os.environ[ARCHIVE_ENV])
DRIVES = ARCHIVE / "2024" / "NOAA_MobileLab_Drives"
WYOMING_GPS = (
    ARCHIVE
    / "2024"
    / "SLC-SOS"
    / "2024_mobile"
    / "Univ_Wyoming"
    / "Mobile_Lab_State_Variables"
    / "ICARTT_GPS"
)
VAN_2026 = ARCHIVE / "2026" / "03_instrument_aligned"
missing = [
    str(tree.relative_to(ARCHIVE)) for tree in (DRIVES, WYOMING_GPS, VAN_2026) if not tree.is_dir()
]
if missing:
    raise RuntimeError(
        f"{ARCHIVE_ENV} is set, but it holds no {missing}. "
        "Point it at the directory containing 2024/ and 2026/."
    )

_work = tempfile.TemporaryDirectory()
WORK = Path(_work.name)
SECOND = 1_000_000_000  # TSARA keeps every time as integer nanoseconds
METRES_PER_DEGREE = 111_320.0

# --- figure style (identical to notebooks 01-04) ----------------------------
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "grid.color": GRID,
        "grid.linewidth": 0.7,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
        "font.size": 9.5,
        "lines.linewidth": 1.4,
        "figure.dpi": 110,
    }
)


def finish(
    ax, title=None, sub=None, ylab=None, xlab=None, legend=False, grid_axis="y", loc="upper right"
):
    """Apply the shared chrome: left-aligned title, muted subtitle, hairline grid."""
    if title:
        ax.set_title(title, pad=19 if sub else 8)
    if sub:
        ax.text(0, 1.025, sub, transform=ax.transAxes, fontsize=8.8, color=MUTED, va="bottom")
    if ylab:
        ax.set_ylabel(ylab)
    if xlab:
        ax.set_xlabel(xlab)
    if grid_axis:
        ax.grid(True, axis=grid_axis, alpha=0.9)
    ax.set_axisbelow(True)
    if legend:
        ax.legend(loc=loc)


def hhmm(ax, fmt="%H:%M"):
    """Label the x axis as clock time; the date belongs in the title."""
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


# --- streams and cells --------------------------------------------------------
def cells_of(stream):
    """Return a stream's cells as CellBounds: integer-nanosecond starts and stops."""
    edges = stream["time_bnds"].values.astype("datetime64[ns]").astype("int64")
    return CellBounds(start_ns=edges[:, 0].copy(), stop_ns=edges[:, 1].copy())


def bounds_of(stream):
    """Return cell starts and stops as datetime64 arrays, for drawing a value across its cell."""
    edges = stream["time_bnds"].values
    return edges[:, 0], edges[:, 1]


def manifest(name, body):
    """Write a manifest into the scratch directory and load it (validated by Pydantic)."""
    path = WORK / f"{name}.yaml"
    path.write_text(body)
    return load_manifest(path)


def angular_difference(a_deg, b_deg):
    """Smallest angle between two directions, in degrees (0 to 180)."""
    return np.abs((np.asarray(a_deg) - np.asarray(b_deg) + 180.0) % 360.0 - 180.0)


# --- reference implementations (share no code with TSARA) --------------------
def reference_bin(reading_cells, values, target, sigma=None, component=None):
    """Overlap-weighted mean onto target cells, one target cell at a time, from the definition.

    overlap_i = max(0, min(stop_i, STOP) - max(start_i, START)); the value is
    sum(overlap_i * x_i) / sum(overlap_i) over readings with a value. With a
    per-reading `sigma`, also returns the propagated sigma: for a "random"
    component sqrt(sum(overlap_i^2 sigma_i^2)) / sum(overlap_i), for a
    "systematic" one sum(overlap_i sigma_i) / sum(overlap_i).
    """
    values = np.asarray(values, dtype=float)
    out = {k: np.full(len(target), np.nan) for k in ("value", "coverage", "sigma")}
    out["n_readings"] = np.zeros(len(target), dtype=int)
    for k in range(len(target)):
        start, stop = int(target.start_ns[k]), int(target.stop_ns[k])
        # Only readings that could overlap: a plain boolean window, for speed.
        near = np.flatnonzero((reading_cells.stop_ns > start) & (reading_cells.start_ns < stop))
        overlap = (
            np.minimum(reading_cells.stop_ns[near], stop)
            - np.maximum(reading_cells.start_ns[near], start)
        ).astype(float)
        use = (overlap > 0) & np.isfinite(values[near])
        if not use.any():
            out["coverage"][k] = 0.0
            continue
        w, rows = overlap[use], near[use]
        out["value"][k] = np.sum(w * values[rows]) / np.sum(w)
        out["n_readings"][k] = rows.size
        out["coverage"][k] = np.sum(w) / (stop - start)
        if sigma is not None:
            s = np.asarray(sigma, dtype=float)[rows]
            if component == "random":
                out["sigma"][k] = np.sqrt(np.sum(w**2 * s**2)) / np.sum(w)
            else:
                out["sigma"][k] = np.sum(w * s) / np.sum(w)
    return out


def reference_vector_mean(reading_cells, angles_deg, target):
    """Overlap-weighted vector mean of directions per target cell: (direction, R)."""
    angles = np.asarray(angles_deg, dtype=float)
    direction, resultant = np.full(len(target), np.nan), np.full(len(target), np.nan)
    for k in range(len(target)):
        start, stop = int(target.start_ns[k]), int(target.stop_ns[k])
        near = np.flatnonzero((reading_cells.stop_ns > start) & (reading_cells.start_ns < stop))
        overlap = (
            np.minimum(reading_cells.stop_ns[near], stop)
            - np.maximum(reading_cells.start_ns[near], start)
        ).astype(float)
        use = (overlap > 0) & np.isfinite(angles[near])
        if not use.any():
            continue
        w = overlap[use] / overlap[use].sum()
        east = np.sum(w * np.sin(np.radians(angles[near][use])))
        north = np.sum(w * np.cos(np.radians(angles[near][use])))
        direction[k], resultant[k] = (
            np.degrees(np.arctan2(east, north)) % 360.0,
            np.hypot(east, north),
        )
    return direction, resultant


# --- checks and the ledger -----------------------------------------------------
CHECKS = {}  # claim -> held?
LEDGER = {}  # claim -> (METHODS section, documented, measured, status)


def check(claim, holds, detail=""):
    """Record one claim about TSARA, compared with a loop from the definition, and print it."""
    CHECKS[claim] = bool(holds)
    print(
        f"  {'✔' if holds else '✘ DOES NOT HOLD:'} {claim}" + (f"   [{detail}]" if detail else "")
    )


def documented(section, claim, value, measured, fmt="{}", applies=True, conditions=""):
    """Put a number METHODS quotes beside the same number re-measured just now.

    Compared as printed, at the precision METHODS prints it. `applies` says
    whether the parameters match the ones METHODS measured under; when they do
    not, the number is shown and not compared.
    """
    shown_doc, shown_now = fmt.format(value), fmt.format(measured)
    status = ("agrees" if shown_doc == shown_now else "DIFFERS") if applies else "not compared"
    LEDGER[claim] = (section, shown_doc, shown_now, status)
    note = "" if applies else f"; METHODS measured {conditions}"
    print(f"  [{status:>12}] {claim}: {shown_now}  (METHODS {section}: {shown_doc}{note})")


# --- logging, with the archive and scratch paths hidden ----------------------
def hide_paths(record):
    """Show the archive and scratch directories as <archive> and <work> in TSARA's log messages."""
    for real, shown in ((str(WORK), "<work>"), (str(ARCHIVE), "<archive>")):
        if isinstance(record.msg, str):
            record.msg = record.msg.replace(real, shown)
        if isinstance(record.args, tuple):
            record.args = tuple(
                str(a).replace(real, shown) if real in str(a) else a for a in record.args
            )
    return True


for handler in setup_logging(logging.WARNING).handlers:
    handler.addFilter(hide_paths)

DRIVE_DAYS = sorted(p.name for p in DRIVES.iterdir() if p.is_dir())
print("archive found; drive days available:", ", ".join(DRIVE_DAYS))

---
## Load one drive

One drive day of the NOAA mobile laboratory, with the instruments sections 1–6
need, read the way a user reads the archive: a manifest, validated, handed to
`ingest_campaign`. Every file on a drive day shares one 1 s merge grid, with a
blank wherever an instrument had nothing to report, so the files look alike and
the instruments do not.

Two declarations are worth reading before the output.

* **`CH4_i_ppb` is loaded as `role: aux`.** The Picarro file carries its methane
  twice: `CH4_ppb` as measured, and `CH4_i_ppb` interpolated into every row.
  TSARA cannot tell them apart, so a manifest naming the `_i` column as a gas
  would feed it exactly the interpolated gas it exists never to produce
  (`METHODS.md` §9.2.3). It is loaded only to be compared.
* **NOy's per-point `NOy_LIF_1SigmaAccuracy`** is the archive's only kind of
  per-point uncertainty column. The file's header says it combines the
  calibration uncertainty (±10 % for NOy) and the zero uncertainty (100 ppt),
  both shared by neighbouring readings, so it is declared **systematic** by
  default. The file writes NOy in pptv, so the manifest converts it, and the
  reported column follows the same scale.

In [ ]:
# ---- PARAMETERS (loading one drive) -----------------------------------------
DRIVE_DAY = "20240718"  # any day printed by the toolkit cell; METHODS measured 20240718
NOY_UNCERTAINTY_COMPONENT = (
    "systematic"  # how to declare the LIF accuracy column: "systematic" or "random"
)

In [ ]:
AT_DOCUMENTED_DAY = DRIVE_DAY == "20240718"  # whether ledger numbers for this drive can be compared
drive = ingest_campaign(
    manifest(
        "one_drive",
        f"""
name: drive_{DRIVE_DAY}
base_path: {DRIVES / DRIVE_DAY}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
      ch4_interpolated: {{column: CH4_i_ppb, role: aux, units: ppb}}
  lif:
    loader: {{format: icartt, path_template: "USOS-NOy-LIF_MobileLab_*.ict"}}
    variables:
      noy:
        column: NOy_LIF
        role: gas
        units: ppb
        convert: {{from_unit: pptv, to_unit: ppb, scale: 0.001}}
        uncertainty:
          {NOY_UNCERTAINTY_COMPONENT}: {{mode: reported, column: NOy_LIF_1SigmaAccuracy}}
  ozone:
    loader: {{format: icartt, path_template: "USOS-O3_MobileLab_*.ict"}}
    variables:
      o3: {{column: O3_ppb, role: gas, units: ppb}}
  ptr:
    loader: {{format: icartt, path_template: "USOS-PTR_MobileLab_*.ict"}}
    variables:
      benzene_ptr: {{column: Benzene_NOAAPTR_ppbv, role: gas, units: ppb}}
  metnav:
    loader: {{format: icartt, path_template: "USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
      wind_dir: {{column: WindDir_calc_deg, role: met, units: degrees, circular: true}}
      wind_speed: {{column: WindSpd_calc_m_s, role: met, units: m s-1}}
      air_temp: {{column: AirTemp_C, role: met, units: degC}}
  iwas:
    loader:
      format: icartt
      path_template: "USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb}}
      toluene_iwas: {{column: Toluene_ppbv, role: gas, units: ppb}}
""",
    )
)
picarro, lif = drive["picarro"], drive["lif"]

print(
    f"\n{'stream':9} {'rows':>7} {'median cell':>12} {'label (where it came from)':>28}  "
    "share of rows holding a value"
)
for name, stream in drive.items():
    width_s = np.median(cells_of(stream).width_ns) / SECOND
    label = (
        f"{stream.attrs['tsara_support_label']} ({stream.attrs['tsara_support_label_provenance']})"
    )
    shares = ", ".join(
        f"{v} {np.isfinite(stream[v].values).mean():.0%}"
        for v in stream.data_vars
        if not str(v).startswith("sigma_")
    )
    print(f"{name:9} {stream.sizes['time']:>7} {width_s:>10.1f} s {label:>28}  {shares}")

---
## 1. The same methane, measured and interpolated

**The question.** Before joining anything: is the column a manifest names the
measurement it claims to be?

The table above answers part of it. Every stream arrives with cells: one second
wide for the merge-grid instruments, and the exact fill intervals for the
canisters, whose file names each fill's stop time. The NOy-LIF file writes
`Time_Mid` and the others `Time_Start`, so TSARA's LIF cells sit half a second
from everyone else's, which matters in section 6.

The other part is below: how often each methane column actually holds a value,
and what the interpolated copy looks like beside the measurement.

In [ ]:
# ---- PARAMETERS (section 1) --------------------------------------------------
WINDOW_S = 120  # seconds of record drawn around the drive's largest measured reading

In [ ]:
documented(
    "§9.2.3",
    "measured Picarro CH₄ rows holding a value",
    0.43,
    float(np.isfinite(picarro["ch4"].values).mean()),
    "{:.0%}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§9.2.3",
    "interpolated CH4_i_ppb rows holding a value",
    1.0,
    float(np.isfinite(picarro["ch4_interpolated"].values).mean()),
    "{:.0%}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§9.2.3",
    "measured O₃ rows holding a value",
    0.50,
    float(np.isfinite(drive["ozone"]["o3"].values).mean()),
    "{:.0%}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

# Where a measured reading exists, how far is the interpolated copy from it?
measured_rows = np.isfinite(picarro["ch4"].values)
copy_gap = np.abs(picarro["ch4_interpolated"].values - picarro["ch4"].values)[measured_rows]
print(
    f"\nwhere a measured reading exists, the _i copy differs from it by a median "
    f"{np.median(copy_gap):.2f} ppb and at most {copy_gap.max():.1f} ppb"
)

# Figure only from here.
peak = pd.Timestamp(picarro["time"].values[int(np.nanargmax(picarro["ch4"].values))])
view = picarro.sel(
    time=slice(peak - pd.Timedelta(seconds=WINDOW_S / 2), peak + pd.Timedelta(seconds=WINDOW_S / 2))
)
has = np.isfinite(view["ch4"].values)
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.plot(
    view["time"].values,
    view["ch4_interpolated"].values,
    color=MUTED,
    lw=1.0,
    label="CH4_i_ppb: a value in every row, interpolated",
)
ax.plot(
    view["time"].values[has],
    view["ch4"].values[has],
    "o",
    color=C2,
    ms=4,
    label="CH4_ppb: what the analyzer reported",
)
finish(
    ax,
    "The same methane, measured and interpolated",
    f"{DRIVE_DAY}, around the drive's largest reading",
    "CH₄ (ppb)",
    "time (UTC)",
    legend=True,
    loc="upper left",
)
hhmm(ax, "%H:%M:%S")
plt.show()

**What to read off the output.** The figure is the trap in one picture. Between
the orange readings, the grey line is a ramp nobody measured, and a manifest
naming `CH4_i_ppb` as a gas would hand TSARA every point on it. Nor is the copy
simply the readings joined up: where a reading exists, the printed difference
says how far the copy sits from it.

**Try it.** `WINDOW_S = 600`: ten minutes. The analyzer reports only every second
or third row throughout, which is what the 43 % in the ledger line means.

---
## 2. Methane on minute cells, and what coverage says

Mirrors notebook 04 §2: the 1 Hz record averaged onto minute cells, with
`n_readings` and `coverage` beside it. The cells come from `grid_cells`, the
epoch-anchored minutes a 60 s grid would use, and the interpolated copy is binned
beside the measured column for contrast.

**The ✔ check** compares both columns with `reference_bin` on every cell of the
drive.

In [ ]:
# ---- PARAMETERS (section 2) --------------------------------------------------
CELL_PERIOD = "60s"

In [ ]:
cells = grid_cells(drive, OutputGridConfig(freq=CELL_PERIOD), ["ch4"])
on_cells = bin_streams_onto_cells(drive, cells, ["ch4", "ch4_interpolated"])
print(f"{len(cells)} cells of {CELL_PERIOD}")
for column in ("ch4", "ch4_interpolated"):
    reference = reference_bin(cells_of(picarro), picarro[column].values, cells)
    check(
        f"§2 {column} on {CELL_PERIOD} cells equals the loop from the definition",
        np.allclose(on_cells[column].values, reference["value"], rtol=1e-12, atol=0, equal_nan=True)
        and np.array_equal(on_cells[f"n_readings_{column}"].values, reference["n_readings"]),
    )
    held = on_cells[f"n_readings_{column}"].values > 0
    n_median = np.median(on_cells[f"n_readings_{column}"].values[held])
    print(
        f"    {column:17} median n_readings {n_median:4.0f}, "
        f"median coverage {np.median(on_cells[f'coverage_{column}'].values[held]):.2f}"
    )
documented(
    "§11.7",
    "median readings per minute of measured CH₄",
    26,
    float(np.median(on_cells["n_readings_ch4"].values)),
    "{:.0f}",
    AT_DOCUMENTED_DAY and CELL_PERIOD == "60s",
    "on 20240718 minutes",
)

# Figure only from here.
rise = pd.Timestamp(on_cells["time"].values[int(np.nanargmax(on_cells["ch4"].values))])
period = pd.Timedelta(CELL_PERIOD)
window = (rise - 4 * period, rise + 5 * period)
win = on_cells.sel(time=slice(*window))
c0, c1 = bounds_of(win)
raw = picarro.sel(time=slice(*window))
finite = np.isfinite(raw["ch4"].values)
fig, (ax, axc) = plt.subplots(
    2, 1, figsize=(9.5, 5.4), sharex=True, gridspec_kw={"height_ratios": [2.3, 1.0], "hspace": 0.3}
)
ax.hlines(
    win["ch4"].values, c0, c1, color=C2, lw=3.4, zorder=2, label=f"binned onto {CELL_PERIOD} cells"
)
ax.plot(
    raw["time"].values[finite],
    raw["ch4"].values[finite],
    ".",
    color=C1,
    ms=3,
    zorder=3,
    label="measured readings",
)
finish(
    ax,
    "Real 1 Hz methane on cells",
    f"{DRIVE_DAY}; the analyzer reports every second or third row",
    "CH₄ (ppb)",
    legend=True,
    loc="upper left",
)
middle = c0 + (c1 - c0) / 2
bar = (c1 - c0) * 0.4
axc.bar(
    middle - bar / 2,
    win["coverage_ch4"].values,
    width=bar,
    color=C1,
    alpha=0.5,
    label="measured column",
)
axc.bar(
    middle + bar / 2,
    win["coverage_ch4_interpolated"].values,
    width=bar,
    color=MUTED,
    alpha=0.5,
    label="interpolated column",
)
axc.set_ylim(0, 1.5)
finish(axc, None, None, "coverage", "time (UTC)")
axc.legend(loc="upper left", ncols=2)
hhmm(axc)
plt.show()

**What to read off the output.** The measured column reports what the analyzer
did: a value in roughly two seconds of every five, so a minute rests on about 26
readings covering well under half of it. The interpolated column reports sixty
readings and full coverage for every minute. Neither is computed wrongly: the
second simply describes interpolated rows, which is why the manifest has to
choose the column.

A reading every 2–3 s inside 1 s cells also means TSARA's inferred cell width for
the Picarro, one second, describes the merge grid rather than the analyzer's own
averaging. That is a width-inference question from Phase 3.5, noted rather than
changed here.

**Try it.** `CELL_PERIOD = "15s"`: a quarter-minute rests on about six readings,
and coverage per cell becomes visibly ragged.

---
## 3. The direction that is refused by default

Mirrors notebook 04 §4. A canister fill is a 15 s mean. Put on the Picarro's 1 s
cells, each fill -- fifteen times as wide as a cell -- would become fifteen rows,
each claiming full coverage (`METHODS.md` §11.2.4). Asked for by name, the copies
are made, labelled, and warned about.

In [ ]:
fill_widest_s = float(cells_of(drive["iwas"]).width_ns.max()) / SECOND
try:
    bin_streams_onto_cells(drive, cells_of(picarro), ["benzene_iwas"])
    refused = False
except TsaraAlignError as refusal:
    refused = True
    print(refusal, "\n")
check(
    "§3 canister fills are refused on 1 s cells exactly when a fill is twice as wide as a cell",
    refused == (fill_widest_s >= 2.0),
    f"widest fill {fill_widest_s:.1f} s",
)

copied = bin_streams_onto_cells(drive, cells_of(picarro), ["benzene_iwas"], finer_support="allow")
record = copied["benzene_iwas"].attrs
n_rows = int(np.isfinite(copied["benzene_iwas"].values).sum())
print(
    f"\nasked for by name: label {record['tsara_support_transform']!r}, "
    f"ratio {record['tsara_width_ratio_max']:.1f}, {n_rows} rows from "
    f"{record['tsara_n_readings']} fills, borrowed share {record['tsara_borrowed_share']:.3f}"
)
check(
    "§3 allowed by name, the column is labelled copied and every fill stands behind about "
    "fifteen rows",
    record["tsara_support_transform"] == "copied"
    and record["tsara_n_readings"] == drive["iwas"].sizes["time"],
)

---
## 4. An uncertainty the file reports

Mirrors notebook 04 §5, with the archive's one per-point uncertainty column.
Declared **systematic**, it is carried through a minute average as the
overlap-weighted mean of the per-reading figures, not reduced by √N. Declared
**random** (the `NOY_UNCERTAINTY_COMPONENT` parameter in *Load one drive*), it
would be reduced. Which is right is a property of the instrument, not of TSARA,
which is why the manifest has to say.

**The ✔ check** recomputes every minute's sigma with the formula for the
declared component.

In [ ]:
noy_minutes = bin_streams_onto_cells(
    drive, grid_cells(drive, OutputGridConfig(freq="60s"), ["ch4"]), ["noy"]
)
name = (
    sigma_sys_name("noy") if NOY_UNCERTAINTY_COMPONENT == "systematic" else sigma_rand_name("noy")
)
sigma = noy_minutes[name]
print(f"the binned sigma column: {name}")
for key in (
    "uncertainty_component",
    "uncertainty_provenance",
    "tsara_sigma_at_support",
    "tsara_propagation_form",
):
    print(f"    {key:24} {sigma.attrs.get(key)}")

reference = reference_bin(
    cells_of(lif),
    lif["noy"].values,
    cells_of(noy_minutes),
    sigma=lif[name].values,
    component=NOY_UNCERTAINTY_COMPONENT,
)
check(
    f"§4 every minute's {NOY_UNCERTAINTY_COMPONENT} sigma equals the formula for that component",
    np.allclose(sigma.values, reference["sigma"], rtol=1e-9, atol=0, equal_nan=True),
)

# Each minute's sigma divided by the plain mean of its own readings' sigmas:
# 1 for a systematic component (no reduction), about 1/sqrt(N) for a random one.
mean_of_readings = reference_bin(
    cells_of(lif),
    lif["noy"].values,
    cells_of(noy_minutes),
    sigma=lif[name].values,
    component="systematic",
)["sigma"]
ratio = np.nanmedian(sigma.values / mean_of_readings)
readings = float(np.nanmedian(noy_minutes["n_readings_noy"].values))
print(
    f"\na minute's sigma ÷ its readings' mean sigma: median {ratio:.3f} "
    f"(1 if nothing is averaged down; 1/√{readings:.0f} = {1 / np.sqrt(readings):.3f} "
    "if it were independent)"
)

**What to read off the output.** Declared systematic, the minute's sigma stays at
the level of its readings, labelled `reported`; a minute holding larger readings
carries a larger figure. That is the default here, because the file's header
describes calibration and zero uncertainties, which neighbouring readings share.

**Try it.** In *Load one drive*, set `NOY_UNCERTAINTY_COMPONENT = "random"` and
re-run it and this section. The printed ratio drops from 1 to near 1/√61: every
minute now claims to be about seven times better determined than its readings,
a precision the file's own header does not support.

---
## 5. Pairing a canister against the analyzer

Mirrors notebook 04 §8: canister benzene against the analyzer's measured
methane. `METHODS.md` §11.4 quotes this pairing on 2024-07-18 and checks it against
a loop from the definition; both are re-run here, together with the obvious
shortcut the loop is meant to rule out.

In [ ]:
# ---- PARAMETERS (section 5) --------------------------------------------------
MIN_COVERAGE = 0.0  # drop pairs whose fill is covered less than this by methane readings

In [ ]:
try:
    canister = pair_species(drive, "benzene_iwas", "ch4", min_coverage=MIN_COVERAGE)
except TsaraAlignError as refusal:
    canister = None
    print(f"min_coverage={MIN_COVERAGE}: {refusal}")

if canister is not None:
    pairs = canister.dataset
    reason = pairs.attrs["tsara_pairing_clock_reason"]
    print(f"clock {canister.clock} ({reason}), {canister.n_pairs} pairs\n")
    widths = {n: float(np.median(cells_of(drive[n]).width_ns)) for n in ("iwas", "picarro")}
    check(
        "§5 the clock is the wider-supported member", canister.clock == max(widths, key=widths.get)
    )
    check(
        "§5 every pair kept meets min_coverage",
        bool(np.all(pairs["coverage_ch4"].values >= MIN_COVERAGE)),
    )

    # The same pairs from the definition, and the obvious shortcut beside them.
    fills = cells_of(pairs)
    reference = reference_bin(cells_of(picarro), picarro["ch4"].values, fills)
    methane, starts = picarro["ch4"].values, cells_of(picarro).start_ns
    # The shortcut: an unweighted mean of the readings whose cells START inside the fill.
    shortcut = np.array(
        [
            np.nanmean(np.where((starts >= a) & (starts < b), methane, np.nan))
            for a, b in zip(fills.start_ns, fills.stop_ns)
        ]
    )
    loop_gap = float(np.max(np.abs(pairs["ch4"].values - reference["value"])))
    check(
        "§5 every pair's methane equals the loop over its fill",
        loop_gap < 1e-9,
        f"largest difference {loop_gap:.1e} ppb",
    )
    applies = AT_DOCUMENTED_DAY and MIN_COVERAGE == 0.0
    documented(
        "§11.4",
        "canister pairs",
        32,
        canister.n_pairs,
        "{}",
        applies,
        "on 20240718 at min_coverage 0",
    )
    documented(
        "§11.4",
        "median coverage of a fill by measured CH₄",
        0.43,
        float(np.median(pairs["coverage_ch4"].values)),
        "{:.2f}",
        applies,
        "on 20240718 at min_coverage 0",
    )
    documented(
        "§11.4",
        "least coverage of a fill",
        0.395,
        float(pairs["coverage_ch4"].values.min()),
        "{:.3f}",
        applies,
        "on 20240718 at min_coverage 0",
    )
    documented(
        "§11.4",
        "median CH₄ readings per fill",
        7,
        float(np.median(pairs["n_readings_ch4"].values)),
        "{:.0f}",
        applies,
        "on 20240718 at min_coverage 0",
    )
    documented(
        "§11.4",
        "largest difference, loop against the shortcut (ppb)",
        38.08,
        float(np.nanmax(np.abs(reference["value"] - shortcut))),
        "{:.2f}",
        applies,
        "on 20240718",
    )

In [ ]:
# Figure only.
if canister is not None:
    k = int(np.nanargmax(pairs["benzene_iwas"].values))
    centre = pd.Timestamp(pairs["time"].values[k])
    span = (centre - pd.Timedelta("150s"), centre + pd.Timedelta("150s"))
    trace = picarro.sel(time=slice(*span))
    ok = np.isfinite(trace["ch4"].values)
    near = pairs.sel(time=slice(*span))
    f0, f1 = bounds_of(near)
    fig = plt.figure(figsize=(10.8, 4.8))
    gs = fig.add_gridspec(
        2, 2, width_ratios=[2.1, 1.0], height_ratios=[1.6, 1.0], hspace=0.35, wspace=0.28
    )
    ax, axs = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[:, 1])
    axb = fig.add_subplot(gs[1, 0], sharex=ax)
    ax.plot(
        trace["time"].values[ok],
        trace["ch4"].values[ok],
        "o",
        color=C1,
        ms=3.2,
        label="measured CH₄ readings",
    )
    for x0, x1 in zip(f0, f1):
        ax.axvspan(x0, x1, color=C3, alpha=0.2, lw=0)
        axb.axvspan(x0, x1, color=C3, alpha=0.2, lw=0)
    ax.hlines(near["ch4"].values, f0, f1, color=C2, lw=4.0, label="CH₄ averaged over each fill")
    finish(
        ax,
        "Real canister fills on the analyzer's record",
        f"{DRIVE_DAY}, around the drive's largest benzene fill",
        "CH₄ (ppb)",
        legend=True,
        loc="upper left",
    )
    axb.hlines(near["benzene_iwas"].values, f0, f1, color=C3, lw=4.0)
    axb.set_ylim(0, float(np.nanmax(near["benzene_iwas"].values)) * 1.3)
    finish(axb, None, None, "benzene (ppb)", "time (UTC)")
    hhmm(axb, "%H:%M:%S")
    points = axs.scatter(
        pairs["ch4"].values,
        pairs["benzene_iwas"].values,
        c=pairs["coverage_ch4"].values,
        cmap="viridis",
        s=26,
    )
    fig.colorbar(points, ax=axs, label="coverage of the fill by CH₄ readings")
    finish(
        axs,
        f"All {canister.n_pairs} fills",
        "one pair per fill",
        "benzene (ppb)",
        "CH₄ (ppb)",
        grid_axis="both",
    )
    plt.show()

**What to read off the output.** TSARA's pairs and the loop agree exactly, and the
shortcut does not: giving an edge reading full weight or none moves a fill's
methane by tens of ppb, against enhancements that are the whole point of a
canister. Coverage says what the figure shows: a fill sampled by half a dozen
readings.

**Try it.**

* `MIN_COVERAGE = 0.9`: TSARA refuses, because no fill on this drive is 90 %
  covered by a methane analyzer that reports every 2–3 s. That is why
  `min_coverage` defaults to zero: a threshold that sounds conservative discards
  the whole canister record.
* `MIN_COVERAGE = 0.42`: about two thirds of the fills survive (22 of 32 on 2024-07-18).

---
## 6. Two 1 s clocks half a second apart

Mirrors notebook 04 §9, on the drive that prompted it. NOy-LIF is dense and
mid-labelled; the Picarro's CO₂ and the ozone monitor are sparse and
start-labelled; the PTR-MS is dense and start-labelled, which makes it the
other case. The code as first written gave a different number of pairs
depending on which species was named first.

In [ ]:
# ---- PARAMETERS (section 6) --------------------------------------------------
PAIRS = [("noy", "co2"), ("noy", "o3"), ("co2", "o3")]  # each is paired in both orders

In [ ]:
for first, second in PAIRS:
    forward, backward = pair_species(drive, first, second), pair_species(drive, second, first)
    reason = forward.dataset.attrs["tsara_pairing_clock_reason"]
    print(
        f"{first} with {second}: clock {forward.clock} ({reason}), {forward.n_pairs} pairs; "
        f"readings {first} {forward.y_readings}, {second} {forward.x_readings}"
    )
    check(
        f"§6 {first}/{second}: swapping the arguments changes neither the clock nor the pair count",
        forward.clock == backward.clock and forward.n_pairs == backward.n_pairs,
    )

# What naming NOy first used to do: the same join on the LIF's cells.
for sparse, instrument in (("co2", "picarro"), ("o3", "ozone")):
    old = bin_streams_onto_cells(drive, cells_of(lif), [sparse, "noy"])
    both = np.isfinite(old[sparse].values) & np.isfinite(old["noy"].values)
    kept = cells_of(old.isel(time=np.flatnonzero(both)))
    reading_cells = cells_of(drive[instrument])
    finite = np.isfinite(drive[instrument][sparse].values)
    # Distinct readings behind those pairs: every reading any pair cell overlaps. The
    # merge-grid cells are sorted and do not overlap each other, so the readings a
    # pair cell [a, b) touches are a contiguous run found by two binary searches.
    touched = np.zeros(len(reading_cells), dtype=bool)
    for a, b in zip(kept.start_ns, kept.stop_ns):
        touched[
            np.searchsorted(reading_cells.stop_ns, a, side="right") : np.searchsorted(
                reading_cells.start_ns, b, side="left"
            )
        ] = True
    distinct = int((touched & finite).sum())
    print(
        f"\non the LIF clock (the old tie-break): {sparse} {int(both.sum())} pairs "
        f"from {distinct} distinct readings"
    )
    if sparse == "co2":
        documented(
            "§11.4.1",
            "NOy/CO₂ pairs on the LIF clock",
            16893,
            int(both.sum()),
            "{}",
            AT_DOCUMENTED_DAY,
            "on 20240718",
        )
        documented(
            "§11.4.1",
            "distinct CO₂ readings behind them",
            8447,
            distinct,
            "{}",
            AT_DOCUMENTED_DAY,
            "on 20240718",
        )
    else:
        documented(
            "§11.4.1",
            "NOy/O₃ pairs on the LIF clock",
            19498,
            int(both.sum()),
            "{}",
            AT_DOCUMENTED_DAY,
            "on 20240718",
        )
on_rule = pair_species(drive, "noy", "co2")
documented(
    "§11.4.1",
    "NOy/CO₂ pairs on the rule's clock",
    8447,
    on_rule.n_pairs,
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§11.4",
    "borrowed share of NOy on the Picarro clock",
    0.50,
    on_rule.dataset["noy"].attrs["tsara_borrowed_share"],
    "{:.2f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

# The same geometry against a DENSE partner: the PTR-MS fills nearly every row.
dense = pair_species(drive, "benzene_ptr", "noy")
shared_ptr = dense.dataset["noy"].attrs["tsara_shared_readings"]
shared_picarro = on_rule.dataset["noy"].attrs["tsara_shared_readings"]
print(
    f"\nbenzene_ptr with noy: clock {dense.clock}, {dense.n_pairs} pairs; NOy readings shared "
    f"between neighbouring pairs {shared_ptr} (against {shared_picarro} on the Picarro clock)"
)
documented(
    "§11.4.1",
    "NOy readings shared between pairs, Picarro clock",
    0,
    on_rule.dataset["noy"].attrs["tsara_shared_readings"],
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§11.4.1",
    "NOy readings shared between pairs, PTR clock",
    18397,
    dense.dataset["noy"].attrs["tsara_shared_readings"],
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
# A NOy reading is shared exactly when BOTH 1 s cells it straddles survived as
# pairs, i.e. when two surviving cells are adjacent -- countable from the product.
for paired in (on_rule, dense):
    adjacent = int((np.diff(paired.dataset["time"].values) == np.timedelta64(1, "s")).sum())
    partner = paired.x_name if paired.y_name == "noy" else paired.y_name
    check(
        f"§6 {partner}: NOy readings shared = adjacent surviving pairs",
        paired.dataset["noy"].attrs["tsara_shared_readings"] == adjacent,
        f"{adjacent}",
    )
documented(
    "§11.4.1",
    "NOy/O₃ pairs on the rule's clock",
    9750,
    pair_species(drive, "noy", "o3").n_pairs,
    "{}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

**What to read off the output.** Both orders now give the sparser member's clock,
and every CO₂ reading is one pair. On the LIF's clock the same readings made
about twice as many pairs, most readings counted in two. Nothing in either run
looks wrong on its own, which is why the clock is now chosen by rule and the
readings are recorded. What remains on the rule's clock is the blend: every NOy
value is the mean of the two readings straddling its cell, the column is
labelled `straddled` with a borrowed share of 0.50, and the warning printed
above says so. It also says the pairs are independent: CO₂ reports every second
or third row, so no NOy reading reaches two pairs (`tsara_shared_readings` = 0),
and no coarser clock is needed. Against the dense PTR benzene the same geometry
shares nearly every NOy reading between neighbouring pairs, and there the
warning names the remedy, `target="10s"` (notebook 04 §9 measures why). The
CO₂/O₃ pair is the other kind of tie: both sparse and both start-labelled, so
the cells coincide and nothing straddles.

**Try it.** Change `DRIVE_DAY` in *Load one drive* to `"20240726"` and re-run it
and this section. Every ✔ still holds on a different day, and the ledger lines
say "not compared", because METHODS measured 2024-07-18.

---
## Load ten drives

Sections 7–10 need all ten 2024 drive days: the Picarro, the MetNav record
(position, ground speed, wind) and the canisters. One manifest covers them, with
`*` in the path template standing for the day directory.

In [ ]:
ten = ingest_campaign(
    manifest(
        "ten_drives",
        f"""
name: ten_drives
base_path: {DRIVES}
platform: {{kind: mobile, gps_instrument: metnav}}
instruments:
  picarro:
    loader: {{format: icartt, path_template: "*/USOS-Picarro-CO2-CH4-CO-H2O_MobileLab_*.ict"}}
    variables:
      co2: {{column: CO2_ppm, role: gas, units: ppm}}
      ch4: {{column: CH4_ppb, role: gas, units: ppb}}
  metnav:
    loader: {{format: icartt, path_template: "*/USOS-MetNav_MobileLab_*.ict"}}
    variables:
      latitude: {{column: GPS_Lat_deg, role: gps_lat, units: degrees_north}}
      longitude: {{column: GPS_Lon_deg, role: gps_lon, units: degrees_east}}
      ground_speed: {{column: GPS_GndSpd_m_s, role: aux, units: m s-1}}
      wind_dir: {{column: WindDir_calc_deg, role: met, units: degrees, circular: true}}
      wind_speed: {{column: WindSpd_calc_m_s, role: met, units: m s-1}}
  iwas:
    loader:
      format: icartt
      path_template: "*/USOS-iWAS_MobileLab_*.ict"
      support: {{stop_column: iWAS_Stop_UTC, method: mean}}
    variables:
      benzene_iwas: {{column: Benzene_ppbv, role: gas, units: ppb}}
""",
    )
)
met = ten["metnav"]
met_starts = cells_of(met).start_ns
# Drives cross midnight UTC, so a drive is a run of rows without an hour's gap, not a calendar date.
drive_number = np.concatenate([[0], np.cumsum(np.diff(met_starts) > 3600 * SECOND)])
fills_all = cells_of(ten["iwas"])
print(
    {name: stream.sizes["time"] for name, stream in ten.items()},
    f"over {drive_number.max() + 1} drives",
)
documented("§11.4", "canister fills on the ten drive days", 261, len(fills_all))
documented(
    "§11.4", "median fill width (s)", 14.9, float(np.median(fills_all.width_ns)) / SECOND, "{:.1f}"
)
fill_drive = drive_number[
    np.clip(np.searchsorted(met_starts, fills_all.start_ns), 0, met_starts.size - 1)
]
between_fills = np.concatenate(
    [np.diff(fills_all.start_ns[fill_drive == n]) for n in np.unique(fill_drive)]
)
documented(
    "§11.4",
    "median time between fills within a drive (s)",
    530,
    float(np.median(between_fills)) / SECOND,
    "{:.0f}",
)

---
## 7. Wind direction on minutes

Mirrors notebook 04 §10. The MetNav's 1 Hz wind direction over ten drives,
vector-averaged onto minutes. The arithmetic mean it replaces is computed with
the same join and the same weights, on a copy of the variable that does not
declare itself circular.

**The ✔ check** compares every minute of the first drive with
`reference_vector_mean`.

In [ ]:
# ---- PARAMETERS (section 7) --------------------------------------------------
WIND_PERIOD = "60s"
WIND_MIN_READINGS = 30  # a cell's direction counts only with at least this many readings

In [ ]:
wind_cells = grid_cells(ten, OutputGridConfig(freq=WIND_PERIOD), ["wind_dir"])
# The same variable twice: once declared circular, once as a plain number.
arithmetic_copy = met.assign(
    wind_dir_arithmetic=("time", met["wind_dir"].values, {"role": "met", "units": "degrees"})
)
wind = bin_streams_onto_cells(
    {"metnav": arithmetic_copy}, wind_cells, ["wind_dir", "wind_dir_arithmetic"]
)

first_drive = met.isel(time=np.flatnonzero(drive_number == 0))
first_cells = cells_of(
    wind.sel(time=slice(first_drive["time"].values[0], first_drive["time"].values[-1]))
)
first_wind = bin_streams_onto_cells({"metnav": first_drive}, first_cells, ["wind_dir"])
reference_direction, reference_r = reference_vector_mean(
    cells_of(first_drive), first_drive["wind_dir"].values, first_cells
)
defined = np.isfinite(reference_direction)
check(
    "§7 the first drive's minute directions and R equal the loop from the definition",
    np.all(
        angular_difference(first_wind["wind_dir"].values[defined], reference_direction[defined])
        < 1e-9
    )
    and np.allclose(
        first_wind["wind_dir_resultant_length"].values, reference_r, atol=1e-12, equal_nan=True
    ),
)

enough = wind["n_readings_wind_dir"].values >= WIND_MIN_READINGS
apart = angular_difference(wind["wind_dir_arithmetic"].values, wind["wind_dir"].values)[enough]
R = wind["wind_dir_resultant_length"].values[enough]
dispersion = wind["wind_dir_dispersion"].values[enough]
applies = WIND_PERIOD == "60s" and WIND_MIN_READINGS == 30
condition = "on 60 s cells with at least 30 readings"
documented(
    "§11.5", "minutes with enough readings", 3337, int(enough.sum()), "{}", applies, condition
)
documented(
    "§11.5",
    "arithmetic mean more than 45° from the vector mean",
    0.267,
    float(np.mean(apart > 45)),
    "{:.1%}",
    applies,
    condition,
)
documented(
    "§11.5",
    "median disagreement (degrees)",
    7.0,
    float(np.median(apart)),
    "{:.1f}",
    applies,
    condition,
)
classes = (
    ("R > 0.99", R > 0.99, 58, 7.1),
    ("R 0.90-0.99", (R > 0.9) & (R <= 0.99), 1140, 18.5),
    ("R 0.50-0.90", (R >= 0.5) & (R <= 0.9), 1713, 40.6),
    ("R < 0.50", R < 0.5, 426, 80.5),
)
for label, members, count_doc, sd_doc in classes:
    documented(
        "§11.5", f"minutes with {label}", count_doc, int(members.sum()), "{}", applies, condition
    )
    documented(
        "§11.5",
        f"median dispersion, {label} (degrees)",
        sd_doc,
        float(np.median(dispersion[members])),
        "{:.1f}",
        applies,
        condition,
    )

In [ ]:
# The first drive only: unit-vector means against speed-weighted means, the
# second built by binning the wind components u and v as ordinary scalars.
speed, direction = first_drive["wind_speed"].values, first_drive["wind_dir"].values
components = first_drive.assign(
    u=("time", speed * np.sin(np.radians(direction)), {"role": "met"}),
    v=("time", speed * np.cos(np.radians(direction)), {"role": "met"}),
)
both_ways = bin_streams_onto_cells({"metnav": components}, first_cells, ["wind_dir", "u", "v"])
weighted = np.degrees(np.arctan2(both_ways["u"].values, both_ways["v"].values)) % 360
usable = (both_ways["n_readings_wind_dir"].values >= WIND_MIN_READINGS) & np.isfinite(weighted)
gap = angular_difference(weighted, both_ways["wind_dir"].values)[usable]
documented(
    "§11.5",
    "first-drive minutes compared, unit against speed-weighted",
    325,
    int(usable.sum()),
    "{}",
    applies,
    condition + " on 20240718",
)
documented(
    "§11.5", "median difference (degrees)", 2.2, float(np.median(gap)), "{:.1f}", applies, condition
)
documented(
    "§11.5", "largest difference (degrees)", 54.6, float(gap.max()), "{:.1f}", applies, condition
)

# Figure only from here: the first full hour of the first drive.
hour_start = pd.Timestamp(first_drive["time"].values[0]).ceil("h")
hour = wind.sel(time=slice(hour_start, hour_start + pd.Timedelta("1h")))
h0, h1 = bounds_of(hour)
raw_hour = first_drive.sel(time=slice(hour_start, hour_start + pd.Timedelta("1h")))
fig, (ax, axr) = plt.subplots(
    2, 1, figsize=(10.0, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.4, 1.0], "hspace": 0.3}
)
ax.plot(
    raw_hour["time"].values,
    raw_hour["wind_dir"].values,
    ".",
    color=C1,
    ms=1.4,
    alpha=0.3,
    label="1 Hz wind direction",
)
ax.hlines(
    hour["wind_dir_arithmetic"].values,
    h0,
    h1,
    color=C3,
    lw=2.2,
    ls=(0, (3, 1.5)),
    label="arithmetic mean of each cell",
)
ax.hlines(hour["wind_dir"].values, h0, h1, color=C2, lw=3.2, label="vector mean (TSARA)")
ax.set_ylim(0, 430)  # headroom above 360 for the legend
ax.set_yticks([0, 90, 180, 270, 360])
finish(
    ax,
    "Real wind direction from a moving van",
    "the first full hour of the first 2024 drive",
    "direction (degrees)",
)
ax.legend(loc="upper left", ncols=3, fontsize=8.2, frameon=True, facecolor=SURFACE, edgecolor=GRID)
axr.plot(hour["time"].values, hour["wind_dir_resultant_length"].values, "o-", color=C2, ms=3)
axr.set_ylim(0, 1.05)
finish(axr, None, None, "resultant length R", "time (UTC)")
hhmm(axr)
plt.show()

**What to read off the output.** On a moving platform a minute's wind direction
is often not a well-determined number, and the lower panel says so minute by
minute; the printed classes count the same thing over ten drives. Part of that
spread is the van turning rather than the air, which TSARA does not try to
separate (`METHODS.md` §11.5). Unit-vector and speed-weighted means differ by a
couple of degrees in a typical minute and by tens of degrees at worst.

**Try it.** `WIND_MIN_READINGS = 55`: only nearly complete minutes count, and the
share of minutes where the arithmetic mean is badly wrong barely moves, so that
failure is not a symptom of sparse minutes.

---
## 8. Positions: attaching the track, and what the guard costs

Mirrors notebook 04 §11. Ingestion left the mobile platform's track on the MetNav
clock; `attach_positions` puts it on the canister fills under the gap guard.

**The ✔ check** recomputes every fill's position by interpolating between the two
MetNav fixes around its midpoint, written out in a loop.

In [ ]:
# ---- PARAMETERS (section 8) --------------------------------------------------
MAX_INTERP_GAP = "10s"
THINNING_STEPS = [5, 10, 30, 50, 60]  # keep every k-th finite fix, for the cost table
MIN_SPEED_M_S = 2.0  # score positions only while the van moves faster than this

In [ ]:
iwas, day_met = drive["iwas"], drive["metnav"]
placed = attach_positions(iwas, drive, max_interp_gap=MAX_INTERP_GAP)
latitude = interpolate_onto_cells(
    drive, ("metnav", "latitude"), cells_of(iwas), max_interp_gap=MAX_INTERP_GAP
)
print("coordinates now on the canister stream:", sorted(map(str, placed.coords)))

# The loop: bracket each fill midpoint by the MetNav fixes around it.
fix_t = cells_of(day_met).midpoint_ns
fix_lat = day_met["latitude"].values
finite_fix = np.isfinite(fix_lat)
fix_t, fix_lat = fix_t[finite_fix], fix_lat[finite_fix]
guard_ns = pd.Timedelta(MAX_INTERP_GAP).value
expected = []
for t in cells_of(iwas).midpoint_ns:
    before, after = np.flatnonzero(fix_t <= t), np.flatnonzero(fix_t >= t)
    if before.size == 0 or after.size == 0:
        expected.append(np.nan)
        continue
    i, j = before[-1], after[0]
    if i != j and fix_t[j] - fix_t[i] > guard_ns:
        expected.append(np.nan)
        continue
    share = 0.0 if i == j else (t - fix_t[i]) / (fix_t[j] - fix_t[i])
    expected.append(fix_lat[i] + share * (fix_lat[j] - fix_lat[i]))
check(
    "§8 every fill's latitude equals the interpolation written out from the fixes",
    np.allclose(placed["latitude"].values, expected, rtol=0, atol=1e-9, equal_nan=True),
)

applies = AT_DOCUMENTED_DAY and MAX_INTERP_GAP == "10s"
documented(
    "§11.6",
    "fills positioned",
    32,
    int(np.isfinite(placed["latitude"].values).sum()),
    "{}",
    applies,
    "on 20240718 with a 10 s guard",
)
documented(
    "§11.6",
    "fills landing exactly on a GPS second",
    2,
    latitude.n_exact,
    "{}",
    applies,
    "on 20240718",
)
documented(
    "§11.6",
    "fills interpolated between two",
    30,
    latitude.n_interpolated,
    "{}",
    applies,
    "on 20240718",
)

# How far the van moves during a fill: the summed path of the fixes inside it.
fix_start = cells_of(day_met).start_ns
lat, lon = day_met["latitude"].values, day_met["longitude"].values
path = []
for start, stop in zip(cells_of(iwas).start_ns, cells_of(iwas).stop_ns):
    inside = (fix_start >= start) & (fix_start <= stop) & np.isfinite(lat)
    steps = np.hypot(
        np.diff(lat[inside]) * METRES_PER_DEGREE,
        np.diff(lon[inside]) * METRES_PER_DEGREE * np.cos(np.radians(40.76)),
    )
    path.append(steps.sum())
documented(
    "§11.6",
    "median path covered during a fill (m)",
    127,
    float(np.median(path)),
    "{:.0f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)
documented(
    "§11.6",
    "longest path covered during a fill (m)",
    307,
    float(np.max(path)),
    "{:.0f}",
    AT_DOCUMENTED_DAY,
    "on 20240718",
)

The thinning table in `METHODS.md` §11.6 is the cost of the guard: every drive's
track, scored only while the van moves, thinned to every *k*-th finite fix,
interpolated linearly, and compared with the removed fixes lying strictly inside a
thinned bracket exactly *k* seconds long.

In [ ]:
lat_all, lon_all, speed_all = (
    met["latitude"].values,
    met["longitude"].values,
    met["ground_speed"].values,
)
seconds_all = met_starts // SECOND
table_doc = {
    5: (117253, 2.8),
    10: (131146, 9.5),
    30: (139222, 54.7),
    50: (140183, 108),
    60: (140335, 137),
}
print(f"{'every k-th fix':>14} {'scored':>8} {'median':>9} {'90th pct':>9} {'largest':>8}")
for k in THINNING_STEPS:
    errors = []
    for number in range(drive_number.max() + 1):
        rows = np.flatnonzero(
            (drive_number == number) & np.isfinite(lat_all) & np.isfinite(lon_all)
        )
        kept = rows[::k]
        kept_t = seconds_all[kept]
        right = np.searchsorted(kept_t, seconds_all[rows], side="right")
        between = (right > 0) & (right < kept.size) & ~np.isin(rows, kept)
        left_fix, right_fix = kept[right[between] - 1], kept[right[between]]
        exact_k = (
            seconds_all[right_fix] - seconds_all[left_fix]
        ) == k  # brackets exactly k seconds long
        target, left_fix, right_fix = rows[between][exact_k], left_fix[exact_k], right_fix[exact_k]
        share = (seconds_all[target] - seconds_all[left_fix]) / k
        guess_lat = lat_all[left_fix] + share * (lat_all[right_fix] - lat_all[left_fix])
        guess_lon = lon_all[left_fix] + share * (lon_all[right_fix] - lon_all[left_fix])
        error = np.hypot(
            (guess_lat - lat_all[target]) * METRES_PER_DEGREE,
            (guess_lon - lon_all[target]) * METRES_PER_DEGREE * np.cos(np.radians(lat_all[target])),
        )
        errors.append(error[speed_all[target] > MIN_SPEED_M_S])
    error = np.concatenate(errors)
    print(
        f"{k:>12} s {error.size:>8} {np.median(error):>7.1f} m "
        f"{np.percentile(error, 90):>7.1f} m {error.max():>6.0f} m"
    )
    if k in table_doc:
        applies = MIN_SPEED_M_S == 2.0
        documented(
            "§11.6",
            f"positions scored at every {k}th fix",
            table_doc[k][0],
            error.size,
            "{}",
            applies,
            "above 2 m/s",
        )
        documented(
            "§11.6",
            f"90th percentile error at every {k}th fix (m)",
            table_doc[k][1],
            float(np.percentile(error, 90)),
            "{:.1f}" if k <= 30 else "{:.0f}",
            applies,
            "above 2 m/s",
        )

And the case where the guard can never be met: the University of Wyoming's mobile
laboratory logged GPS as a separate file per drive, several of them a fix every
50 s. Against the default 10 s guard, TSARA warns.

In [ ]:
# ---- PARAMETERS (section 8, the Wyoming GPS logs) ---------------------------
WYOMING_GUARD = "10s"

In [ ]:
spacing, positioned = {}, {}
auxiliary_log, ingest_log = (
    logging.getLogger("tsara.align.auxiliary"),
    logging.getLogger("tsara.ingest"),
)
ingest_log.setLevel(
    logging.ERROR
)  # some logs repeat a timestamp, reported file by file; not this cell's subject
for path in sorted(WYOMING_GPS.glob("UWMobileLab_GPSdata_*.ict")):
    gps = ingest_campaign(
        manifest(
            "wyoming_gps",
            f"""
name: wyoming_gps
base_path: {WYOMING_GPS}
platform: {{kind: stationary, latitude: 40.76, longitude: -111.89}}
instruments:
  gps:
    loader: {{format: icartt, path_template: "{path.name}"}}
    variables:
      gps_lat: {{column: LATITUDE, role: gps_lat, units: degrees_north}}
""",
        )
    )["gps"]
    fix_ns = cells_of(gps).start_ns[np.isfinite(gps["gps_lat"].values)]
    spacing[path.name] = float(np.median(np.diff(fix_ns))) / SECOND
    whole = np.arange(fix_ns[0] // SECOND * SECOND, fix_ns[-1], SECOND)
    # One file's warning is shown; the others would repeat it.
    auxiliary_log.setLevel(logging.NOTSET if "_D06_" in path.name else logging.ERROR)
    field = interpolate_onto_cells(
        {"gps": gps},
        "gps_lat",
        CellBounds(start_ns=whole, stop_ns=whole + SECOND),
        max_interp_gap=WYOMING_GUARD,
    )
    positioned[path.name] = float(np.isfinite(field.values).mean())
auxiliary_log.setLevel(logging.NOTSET)
ingest_log.setLevel(logging.NOTSET)
for name in spacing:
    print(
        f"  {name:42} a fix every {spacing[name]:5.1f} s; "
        f"1 s cells positioned {positioned[name]:6.1%}"
    )
fifty = [name for name in spacing if spacing[name] >= 50]
documented("§11.6", "Wyoming GPS files logging every 50 s", 7, len(fifty))
documented(
    "§11.6",
    "1 s cells those files position at the guard",
    0.0,
    max(positioned[name] for name in fifty),
    "{:.1%}",
    WYOMING_GUARD == "10s",
    "with a 10 s guard",
)

**What to read off the output.** A record sampled every 50 s cannot meet a 10 s
guard anywhere, so the join positions nothing. Until walkthrough stage 5 it said
so only at INFO level, which a notebook never shows.

The 10 s logs position only about a third of their 1 s cells, which looks like the
same problem and is not. Their fixes do arrive every 10 s, but the logging stops
for about 410 s at a time, repeatedly (about two hundred such gaps in the first
file), and the guard correctly refuses to bridge those. Raising the guard to 30 s
positions less than one percent more of the cells.

**Try it.**

* `MAX_INTERP_GAP = "0.5s"`: shorter than the MetNav's 1 s spacing, so only the
  fills whose midpoints land exactly on a GPS second keep a position. The ✔ still
  holds, because the loop applies the same guard. (A guard of exactly `"1s"`
  still bridges every gap: a gap equal to the guard is allowed.)
* `WYOMING_GUARD = "60s"`: the 50 s logs are now positioned about as fully as the
  10 s logs, a third of their cells, the rest being the same long logging gaps.
  The thinning table above says what bridging 50 s costs: about 110 m at the 90th
  percentile, at city speeds.

---
## 9. The drive's matrix

Mirrors notebook 04 §12, with the selection `METHODS.md` §11.7 states: the
Picarro's measured CO₂ and CH₄, MetNav wind direction and air temperature,
NOy-LIF, PTR-MS benzene, and the canisters' benzene and toluene.

**The ✔ check** recomputes every column of the matrix with `reference_bin` (and
`reference_vector_mean` for wind).

In [ ]:
# ---- PARAMETERS (section 9) --------------------------------------------------
MATRIX_PERIOD = "60s"
SELECTION = [
    "co2",
    "ch4",
    "wind_dir",
    "air_temp",
    "noy",
    "benzene_ptr",
    "benzene_iwas",
    "toluene_iwas",
]

In [ ]:
DEFAULT_SELECTION = [
    "co2",
    "ch4",
    "wind_dir",
    "air_temp",
    "noy",
    "benzene_ptr",
    "benzene_iwas",
    "toluene_iwas",
]
at_documented = AT_DOCUMENTED_DAY and SELECTION == DEFAULT_SELECTION
without_canisters = [v for v in SELECTION if not v.endswith("_iwas")]
for freq, chosen, rows_doc in (
    ("5s", SELECTION, None),
    ("15s", SELECTION, 1302),
    ("1s", without_canisters, 19502),
):
    try:
        rows = len(grid_cells(drive, OutputGridConfig(freq=freq), chosen))
        print(f"{freq:>4} over {len(chosen)} variables: {rows} cells")
        if rows_doc is not None:
            documented(
                "§11.7",
                f"{freq} grid cells",
                rows_doc,
                rows,
                "{}",
                at_documented,
                "on 20240718, default selection",
            )
    except TsaraAlignError as refusal:
        print(f"{freq:>4} over {len(chosen)} variables: refused: {str(refusal).split('. ')[0]}.")

matrix = build_output_grid(drive, OutputGridConfig(freq=MATRIX_PERIOD), SELECTION)
matrix_cells = cells_of(matrix)
columns_ok = True
for variable in SELECTION:
    instrument = matrix[variable].attrs["tsara_instrument"]
    stream = drive[instrument]
    if variable == "wind_dir":
        direction, _ = reference_vector_mean(
            cells_of(stream), stream[variable].values, matrix_cells
        )
        defined = np.isfinite(direction)
        columns_ok &= bool(
            np.all(angular_difference(matrix[variable].values[defined], direction[defined]) < 1e-9)
        )
    else:
        reference = reference_bin(cells_of(stream), stream[variable].values, matrix_cells)
        columns_ok &= np.allclose(
            matrix[variable].values, reference["value"], rtol=1e-12, atol=0, equal_nan=True
        )
check(
    f"§9 every column of the {MATRIX_PERIOD} matrix equals the loop from the definition", columns_ok
)

share_doc = {
    "co2": (0.994, 26),
    "ch4": (0.994, 26),
    "wind_dir": (0.997, 60),
    "air_temp": (0.997, 60),
    "noy": (1.0, 61),
    "benzene_ptr": (0.972, None),
    "benzene_iwas": (0.113, 0),
    "toluene_iwas": (0.113, 0),
}
print(
    f"\n{'variable':13} {'rows filled':>12} {'median n_readings':>18} {'readings':>9} "
    f"{'label':>11} {'ratio':>6} {'borrowed':>9}"
)
for variable in SELECTION:
    counts = matrix[f"n_readings_{variable}"].values
    a = matrix[variable].attrs
    print(
        f"{variable:13} {np.mean(counts > 0):>12.1%} {np.median(counts):>18.0f} "
        f"{a['tsara_n_readings']:>9} {a['tsara_support_transform']:>11} "
        f"{a['tsara_width_ratio_max']:>6.3f} {a['tsara_borrowed_share']:>9.3f}"
    )
    if variable in share_doc:
        applies = at_documented and MATRIX_PERIOD == "60s"
        documented(
            "§11.7",
            f"60 s rows holding {variable}",
            share_doc[variable][0],
            float(np.mean(counts > 0)),
            "{:.1%}",
            applies,
            "on 20240718 minutes",
        )
        if share_doc[variable][1] is not None:
            documented(
                "§11.7",
                f"median n_readings of {variable}",
                share_doc[variable][1],
                float(np.median(counts)),
                "{:.0f}",
                applies,
                "on 20240718 minutes",
            )

**What to read off the output.** The matrix is honest about each instrument. Met
and the LIF fill almost every minute from about sixty readings; the Picarro fills
them from about 26; the canisters appear in about one row in nine, and the
warning above names them because some fills straddle a minute boundary.

The LIF's half-second offset does not matter to a minute row, which is far wider
than its cells, but its cells are exactly the period of a one-second grid:

In [ ]:
one_second = build_output_grid(drive, OutputGridConfig(freq="1s"), ["co2", "noy"])
counts, how_many = np.unique(one_second["n_readings_noy"].values, return_counts=True)
print("n_readings of noy on the 1 s grid:", dict(zip(counts.tolist(), how_many.tolist())))

Every LIF value on that grid is the mean of two neighbouring readings. Starting
the grid on the LIF's own boundaries would do the same to the Picarro. Which
instrument a 1 s grid should respect is the user's decision, made with `start`.

Notebook 04 §13 shows why a joined product may not be joined again: its rows are
not readings, so a second pass would weight and count them as though they were,
and on this drive a five-minute row built from the minute matrix would report
its CH₄ fully covered where the stream says 0.43. So the matrix is refused as an
input, and the five-minute rows are built from the stream:

In [ ]:
try:
    build_output_grid({"matrix": matrix}, OutputGridConfig(freq="300s"), ["ch4"])
    rejoined = True
except TsaraAlignError as refusal:
    rejoined = False
    print(f"the matrix as an input: {str(refusal).split('. ')[0]}.\n")
check("§9 a product this package built may not be joined again", not rejoined)
five_from_stream = build_output_grid(drive, OutputGridConfig(freq="300s"), ["ch4"])
occupied = five_from_stream["n_readings_ch4"].values > 0
print(
    f"median coverage of a 5-minute row by measured CH₄, built from the stream: "
    f"{np.median(five_from_stream['coverage_ch4'].values[occupied]):.2f}"
)

**Try it.** `SELECTION = ["co2", "ch4", "noy"]` with `MATRIX_PERIOD = "2s"`: no
canisters, so a fine matrix is allowed, and the ✔ still holds.

---
## 10. Ten drives on one grid

Mirrors notebook 04 §12. A uniform grid spans the weeks between drives, which is
where its size and its compressibility come from. Then the measurement behind the
deleted median option (`METHODS.md` §11.7.1), under its stated rule.

In [ ]:
# ---- PARAMETERS (section 10) -------------------------------------------------
CAMPAIGN_PERIODS = ["60s", "1s"]
COMPRESSION_LEVEL = 4
BASELINE_WINDOW = "600s"  # the median rule's rolling baseline window ...
BASELINE_QUANTILE = 0.05  # ... and quantile
ENHANCED_ABOVE_PPB = 5.0  # a minute counts as enhanced above this mean enhancement

In [ ]:
THREE = ["co2", "ch4", "wind_dir"]
rows_doc = {"60s": 42443, "1s": 2546521}
campaign_grid = None
for freq in CAMPAIGN_PERIODS:
    campaign_grid = build_output_grid(ten, OutputGridConfig(freq=freq), THREE)
    any_data = np.zeros(campaign_grid.sizes["time"], dtype=bool)
    for variable in THREE:
        any_data |= campaign_grid[f"n_readings_{variable}"].values > 0
    grid = cells_of(campaign_grid)
    span_ns = int(grid.stop_ns[-1] - grid.start_ns[0])
    print(
        f"{freq:>4}: {campaign_grid.sizes['time']:,} rows over {span_ns / 86400e9:.1f} days, "
        f"{any_data.mean():.1%} holding any data, {campaign_grid.nbytes / 1e6:.0f} MB"
    )
    check(
        f"§10 the {freq} campaign grid's rows are its span divided by its period",
        campaign_grid.sizes["time"] == span_ns // pd.Timedelta(freq).value,
    )
    if freq in rows_doc:
        documented(
            "§11.7", f"{freq} ten-drive grid rows", rows_doc[freq], campaign_grid.sizes["time"]
        )
    if freq == "1s":
        documented("§11.7", "1 s ten-drive rows empty", 0.921, float(1 - any_data.mean()), "{:.1%}")

sizes, identical = {}, True
for level in (None, COMPRESSION_LEVEL):
    written = save_grid(campaign_grid, WORK / f"ten_drives_{level}", compression=level)
    sizes[level] = written.stat().st_size / 1e6
    back = load_grid(WORK / f"ten_drives_{level}")
    identical &= all(
        np.array_equal(campaign_grid[v].values, back[v].values, equal_nan=True)
        for v in campaign_grid.data_vars
    )
    print(
        f"the {CAMPAIGN_PERIODS[-1]} grid on disk, compression={level!s:>4}: {sizes[level]:6.1f} MB"
    )
check("§10 the campaign grid reloads identically, compressed or not", identical)
if CAMPAIGN_PERIODS[-1] == "1s":
    documented(
        "§11.7", "1 s ten-drive grid on disk, uncompressed (MB)", 346.4, sizes[None], "{:.1f}"
    )
    documented(
        "§11.7",
        "1 s ten-drive grid on disk, compression 4 (MB)",
        3.8,
        sizes[COMPRESSION_LEVEL],
        "{:.1f}",
        COMPRESSION_LEVEL == 4,
        "at level 4",
    )
del campaign_grid

with_canisters = build_output_grid(ten, OutputGridConfig(freq="60s"), THREE + ["benzene_iwas"])
rows60 = cells_of(with_canisters)
first_row = (fills_all.start_ns - rows60.start_ns[0]) // (60 * SECOND)
last_row = (fills_all.stop_ns - 1 - rows60.start_ns[0]) // (60 * SECOND)
documented(
    "§11.7",
    "60 s rows holding canister benzene",
    320,
    int((with_canisters["n_readings_benzene_iwas"].values > 0).sum()),
)
documented("§11.7", "fills landing in two rows", 68, int((last_row - first_row == 1).sum()))
documented(
    "§11.7",
    "borrowed share of canister benzene on 60 s rows",
    0.10,
    with_canisters["benzene_iwas"].attrs["tsara_borrowed_share"],
    "{:.2f}",
)

# The same ten drives on a 15 s grid: the canisters' longest fills are wider than
# a cell, so the column is narrowed -- allowed, labelled, and said aloud.
fifteen = build_output_grid(ten, OutputGridConfig(freq="15s"), ["ch4", "benzene_iwas"])
narrowed = fifteen["benzene_iwas"].attrs
rows15 = int((fifteen["n_readings_benzene_iwas"].values > 0).sum())
print(
    f"\ncanister benzene on a 15 s grid: label {narrowed['tsara_support_transform']!r}, ratio "
    f"{narrowed['tsara_width_ratio_max']:.2f}, {rows15} rows from "
    f"{narrowed['tsara_n_readings']} fills"
)
documented(
    "§11.7", "widest canister fill (s)", 20.1, float(fills_all.width_ns.max()) / SECOND, "{:.1f}"
)
documented(
    "§11.7",
    "widest fill over a 15 s cell (width ratio)",
    1.34,
    narrowed["tsara_width_ratio_max"],
    "{:.2f}",
)
documented(
    "§11.7",
    "15 s rows holding canister benzene",
    520,
    int((fifteen["n_readings_benzene_iwas"].values > 0).sum()),
)
check(
    "§10 on a 15 s grid the canister column is narrowed exactly when its widest fill is wider "
    "than a cell",
    (narrowed["tsara_support_transform"] == "narrowed")
    == (float(fills_all.width_ns.max()) > 15 * SECOND),
)

In [ ]:
# METHODS §11.7.1, under its stated rule: each reading's enhancement is its value
# minus a rolling BASELINE_QUANTILE of the readings over BASELINE_WINDOW; readings
# are grouped into epoch-aligned minutes; a minute is enhanced when its mean
# enhancement exceeds ENHANCED_ABOVE_PPB; a negative median counts as zero.
picarro_all = ten["picarro"]
series = pd.Series(
    picarro_all["ch4"].values, index=pd.DatetimeIndex(cells_of(picarro_all).start_ns)
)
enhanced = []
for _, one_drive in series.groupby(drive_number):
    baseline = one_drive.rolling(BASELINE_WINDOW, min_periods=60).quantile(BASELINE_QUANTILE)
    excess = (one_drive - baseline).dropna()
    by_minute = excess.groupby(excess.index.floor("60s"))
    table = pd.DataFrame(
        {
            "mean": by_minute.mean(),
            "median": by_minute.median(),
            "above": by_minute.apply(lambda x: float((x > ENHANCED_ABOVE_PPB).mean())),
        }
    )
    enhanced.append(table[table["mean"] > ENHANCED_ABOVE_PPB])
enhanced = pd.concat(enhanced)
kept = enhanced["median"].clip(lower=0)
thin = enhanced["above"] < 0.5
zeroed = enhanced["median"] <= 0
applies = BASELINE_WINDOW == "600s" and BASELINE_QUANTILE == 0.05 and ENHANCED_ABOVE_PPB == 5.0
rule = "under the stated rule"
discarded = 1 - kept.sum() / enhanced["mean"].sum()
print(
    f"{len(enhanced)} enhanced minutes; a 60 s median discards {discarded:.1%} of their enhancement"
)
documented("§11.7.1", "enhanced minutes", 2147, len(enhanced), "{}", applies, rule)
documented(
    "§11.7.1",
    "enhancement mass a 60 s median discards",
    0.196,
    float(1 - kept.sum() / enhanced["mean"].sum()),
    "{:.1%}",
    applies,
    rule,
)
documented(
    "§11.7.1",
    "minutes with fewer than half their readings enhanced",
    334,
    int(thin.sum()),
    "{}",
    applies,
    rule,
)
documented(
    "§11.7.1",
    "what the median loses over those minutes",
    0.788,
    float(1 - kept[thin].sum() / enhanced["mean"][thin].sum()),
    "{:.1%}",
    applies,
    rule,
)
documented(
    "§11.7.1", "minutes a median reduces to zero", 20, int(zeroed.sum()), "{}", applies, rule
)
documented(
    "§11.7.1",
    "largest of them (ppb)",
    42.8,
    float(enhanced["mean"][zeroed].max()),
    "{:.1f}",
    applies,
    rule,
)

**What to read off the output.** On real drive methane a 60 s median discards about a
fifth of the enhancement, concentrated in minutes a plume only partly fills: in the
minutes where fewer than half the readings are enhanced it loses nearly four
fifths. That is the measurement behind deleting the median option.

**Try it.**

* `ENHANCED_ABOVE_PPB = 20.0`: only strong minutes count, and less than half as
  many qualify, yet the share a median discards barely moves (about 19 %). The
  loss is not an artefact of counting weak minutes. The ledger lines say "not
  compared", because the rule changed.
* `CAMPAIGN_PERIODS = ["60s"]`: skip the one-second grid; the disk sizes are then
  for the 60 s grid, and the one-second size lines are not written.

---
## 11. The 2026 van: an analyzer that is not quite one second

Mirrors the "too fine" part of notebook 04 §12. Two LANL Aeris methane analyzers
on one 2026 drive, from the aligned parquet stage, with the quarantine directories
excluded.

In [ ]:
# ---- PARAMETERS (section 11) -------------------------------------------------
VAN_DAY = "260119"  # a yymmdd present in the LANL Aeris file names; METHODS measured 260119
VAN_GRID_PERIOD = "1s"

In [ ]:
van = ingest_campaign(
    manifest(
        "van",
        f"""
name: van_{VAN_DAY}
base_path: {VAN_2026}
platform: {{kind: stationary, latitude: 40.76, longitude: -111.89}}
instruments:
  pico:
    loader:
      format: parquet
      path_template: "LANL_aerispico017/Eng/Pico100017_{VAN_DAY}_*Eng.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4_pico: {{column: CH4_ppm, role: gas, units: ppm}}
  ultra:
    loader:
      format: parquet
      path_template: "LANL_aerisultra321/Eng/Ultra100321_{VAN_DAY}_*Eng.parquet"
      exclude: ["**/bad/**", "**/bad_timestamp/**"]
    variables:
      ch4_ultra: {{column: CH4_ppm, role: gas, units: ppm}}
""",
    )
)
widest = {}
for name, stream in van.items():
    widths = cells_of(stream).width_ns / SECOND
    widest[name] = float(widths.max())
    print(
        f"{name}: {stream.sizes['time']} cells, median width {np.median(widths):.3f} s, "
        f"widest {widths.max():.3f} s"
    )
at_van_day = VAN_DAY == "260119"
documented(
    "§11.7",
    "Aeris pico cell width (s)",
    1.023,
    float(np.median(cells_of(van["pico"]).width_ns)) / SECOND,
    "{:.3f}",
    at_van_day,
    "on 260119",
)
period_s = pd.Timedelta(VAN_GRID_PERIOD).total_seconds()
try:
    van_grid = build_output_grid(van, OutputGridConfig(freq=VAN_GRID_PERIOD))
    built = True
except TsaraAlignError as refusal:
    van_grid, built = None, False
    print(f"refused: {str(refusal).split('. ')[0]}.")
check(
    f"§11 a {VAN_GRID_PERIOD} grid is built exactly when no cell is twice as wide as a grid cell",
    built == all(w < 2 * period_s for w in widest.values()),
)
if van_grid is not None:
    occupied = int((van_grid["n_readings_ch4_pico"].values > 0).sum())
    a = van_grid["ch4_pico"].attrs
    print(
        f"ch4_pico: {occupied} rows from {a['tsara_n_readings']} readings; "
        f"label {a['tsara_support_transform']!r}, ratio {a['tsara_width_ratio_max']:.3f}, "
        f"borrowed share {a['tsara_borrowed_share']:.2f}"
    )
    at_one_second = at_van_day and VAN_GRID_PERIOD == "1s"
    documented(
        "§11.7",
        "1 s rows holding ch4_pico",
        17517,
        occupied,
        "{}",
        at_one_second,
        "on 260119 at 1 s",
    )
    documented(
        "§11.7",
        "ch4_pico readings behind them",
        17069,
        a["tsara_n_readings"],
        "{}",
        at_one_second,
        "on 260119 at 1 s",
    )
    documented(
        "§11.7",
        "ch4_pico width ratio on 1 s cells",
        1.024,
        a["tsara_width_ratio_max"],
        "{:.3f}",
        at_one_second,
        "on 260119 at 1 s",
    )
    documented(
        "§11.7",
        "ch4_pico borrowed share on 1 s cells",
        0.34,
        a["tsara_borrowed_share"],
        "{:.2f}",
        at_one_second,
        "on 260119 at 1 s",
    )

**What to read off the output.** A one-second grid is built for an analyzer
reporting 1.023 s cells, labelled `narrowed` with a ratio of 1.024 -- which is
what a 1.023 s reading on a 1 s cell is -- with a few percent more rows than
readings and a third of every value borrowed from the neighbouring cell; the
warning names both analyzers with those numbers. The comparison of median
widths that walkthrough stage 6 replaced would have refused the grid outright.

**Try it.** `VAN_GRID_PERIOD = "0.5s"`: refused, because a 1.023 s cell is twice
as wide as a half-second cell.

---
## 12. The ledger and the scoreboard

Every archive number this notebook re-measured beside what `METHODS.md` says, each
compared at the precision METHODS prints it; then every ✔ check. If you changed
parameters, the ledger shows which numbers could not be compared.

In [ ]:
ledger = pd.DataFrame(
    [(claim, *row) for claim, row in LEDGER.items()],
    columns=["claim", "METHODS", "documented", "measured", "status"],
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 70, "display.width", 200):
    print(ledger[["METHODS", "claim", "documented", "measured", "status"]].to_string(index=False))
counts = ledger["status"].value_counts()
print(
    f"\nledger: {counts.get('agrees', 0)} agree, {counts.get('DIFFERS', 0)} differ, "
    f"{counts.get('not compared', 0)} not compared, of {len(ledger)}"
)
print(f"checks: {sum(CHECKS.values())} of {len(CHECKS)} hold")
for claim, held in CHECKS.items():
    if not held:
        print(f"  ✘ {claim}")